# Demo: part B's checks on one random scene

This notebook takes **one random scene** (fixed `SEED`) from Zach's `suite_v1` prompt suite, plus two country items (the positive control), and runs on them, step by step, the checks that claims C1 and C2 of the v3 plan rest on:

| § | Check | Question | Zach's script | EXPERIMENTS.md row |
|---|---|---|---|---|
| 2 | competence | does the model solve the task, and each step alone? | `scripts/suite_v1.py` | 24 |
| 3 | lens read | does J++ see the bridge at the question tokens? | `scripts/read_write.py` | 32 |
| 4 | patch map | at which layers and positions does the model carry the bridge? | `scripts/suite_split.py` (map) | 27 |
| 5 | patch split | which part of the patch moves the answer: the J-space part or the rest? | `scripts/suite_split.py` (parts) | 27, 31 |
| 6 | read vs write | can a push along the lens direction *write* the bridge? | `scripts/read_write.py` | 37, 52 |
| 7 | knock-out | does the task need the J-space at all? | `scripts/ablate_pilot.py` | 42, 55 |

Each section first shows the mechanics by hand on a single edit, then runs the full set of edits as the script does, and **compares with the rows in Zach's `results/` for the same items**. The seeds are the same, so the numbers should agree up to bf16 rounding in batched forwards.

What this notebook does **not** do: it writes nothing to `results/` and produces no headline numbers. One item illustrates the mechanism; the headlines rest on n=60–150 in Zach's runs and are still UNVERIFIED (none is in VERIFIED.md).

**Running it.** A GPU with ~80 GB for Qwen3.6-27B (`BJ_TARGET=qwen3.5-9b` for a smaller one). On a new box: `bash infra/setup_pod.sh`, the model and J++ in the HF cache, then open this file in VS Code Remote-SSH with the `.venv` kernel. Loading the model takes under a minute from a local disk, the rest about 5 minutes.

### The task, and what the names mean

Every question needs two steps: **step 1** finds an intermediate entity, the **bridge**; **step 2** uses it to get the **answer**. Each step takes its fact either from the **prompt** or from the model's **memory**. The four families of the animal suite cross the two, on one and the same context ("Oscar's pet is a spider. … The spider's locker is 9. …"):

| Family | Step 1: find the bridge | Step 2: use it | Example | Zach's name |
|---|---|---|---|---|
| **prompt → prompt** | read it in the prompt | look its locker up in the prompt | Which locker belongs to Oscar's pet? | BL (bound → lookup) |
| **prompt → memory** | read it in the prompt | recall its legs | How many legs does Oscar's pet have? | BR (bound → recall) |
| **memory → prompt** | recall which animal it is | look its locker up in the prompt | Which locker belongs to the animal known for spinning webs? | RL (recalled → lookup) |
| **memory → memory** | recall which animal it is | recall its legs | How many legs does the animal known for spinning webs have? | RR (recalled → recall) |

The positive control (exp09, control_split) uses countries, where step 2 is always recalling a capital: **countries: memory → memory** ("the capital of the country where Lyon is located", Zach's R) and **countries: prompt → memory** ("Bob lives in France. … the capital of the country where Bob lives", Zach's B).

The claim under test (C1/C2): a bridge used by a step from **memory** goes through the J-space; one used by a step from the **prompt** goes around it.

Every item comes in three prompts that line up token for token, so residuals can be copied position by position:

| Here | Zach's name | What it is |
|---|---|---|
| **clean** prompt | clean | the question as asked; its answer A_org (9) |
| **swap** prompt | C (counterfactual) | only the bridge changes, B_org → B_alt (spider → pig), so the answer becomes A_alt (9 → 5) |
| **control** prompt | O (off-path) | changes something the answer does not depend on (two other lockers swapped) |
| **B_org** / **B_alt** | I / X | the bridge in the clean prompt / in the swap prompt |
| **A_org** / **A_alt** | A / b | the answer to the clean prompt / to the swap prompt |
| **patch** | copy | add `h_swap − h_clean` (one layer, some positions) to the clean run |
| **shift** | frac | how far an edit moves the answer toward A_alt: 0% = clean, 100% = as the swap prompt |
| **flip** | flip (role b) | the top token becomes A_alt |
| **carry window** | parts window | layers 32–40: the patch at the question tokens moves the answer there, before the last token holds it |
| **effect ÷ size** | concentration | a part's shift over what its share of the patch predicts |

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("BJ_TARGET", "qwen3.6-27b")  # or "qwen3.5-9b" (18 GB): set before bridge_jspace is imported
os.environ.setdefault("BJ_BATCH", "32")  # edited prompts per batched forward (Zach ran 64 on 141 GB H200s)
SEED = 0  # picks the scene and the two country items; change it to see others
WRITE_EARLY = True  # §6: also push at early layers (EXPERIMENTS.md row 52); 27B only

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]  # `scripts.*` import from the repo root, as in Zach's scripts

In [ ]:
import collections
import json
import math
import random
import statistics
import textwrap
import time

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from bridge_jspace import pins
from bridge_jspace.plotting import INK, MUTED, SURFACE, percent_axis, shade_band, style_axes
from scripts import ablate_pilot as AB
from scripts import exp09_bind_recall as E
from scripts import suite_split as P
from scripts import suite_v1 as S
from scripts.exp04_brew_parts import K_ATOMS, AddRows, lens2_part, null_rows, signed_omp

print(f"{pins.TARGET}: {pins.MODEL_REPO}, {pins.N_LAYERS} layers; batch {P.BATCH}")

In [ ]:
RUNS = {  # Zach's runs this notebook checks itself against (EXPERIMENTS.md rows in brackets)
    "qwen3.6-27b": {
        "competence": "competence_v2",  # [24] every prompt of every item, unedited
        "map": "map_A8_v1",  # [27] patch map; its 150 scenes are the cases of every later run
        "parts": "parts_A8_cal_v1",  # [31] patch split, with the random-dictionary null and the bridge-word part
        "read_write": "dose_v1",  # [37] lens read and write at 32/36/40, 0.5-16x
        "read_write_early": "layers_v1",  # [52] the same at 16-28 (and 44-52)
        "ablate": "abl_rot_v1",  # [55] J-space knock-out with the rotated control
        "country_map": "map_v1",  # [30] control_split's country cases (the positive control)
        "window": (32, 36, 40),  # the carry window: read/write and knock-out layers
        "early": (16, 20, 24, 28),
        "country_window": (36, 40),  # control_split parts_v3: the question-token window
    },
    "qwen3.5-9b": {
        "competence": "qwen35_9b_competence_v2",
        "map": "qwen35_9b_map_A8_v1",
        "parts": "qwen35_9b_parts_A8_v1",
        "read_write": "qwen35_9b_rw_v1",
        "read_write_early": None,
        "ablate": None,  # no 9B knock-out run: §7 runs with nothing to compare against
        "country_map": None,
        "window": (16, 18, 20),
        "early": (),
        "country_window": (),
    },
}
R = RUNS[pins.TARGET]
CORE = ("BL", "BR", "RL", "RR")  # family keys stay Zach's: his item files and results use them
COUNTRIES = ("R", "B") if R["country_map"] else ()
FAMILIES = CORE + COUNTRIES
LABEL = {
    "BL": "prompt → prompt",
    "BR": "prompt → memory",
    "RL": "memory → prompt",
    "RR": "memory → memory",
    "R": "countries: memory → memory",
    "B": "countries: prompt → memory",
}
SCALES = (0.5, 1.0, 2.0, 4.0, 8.0, 16.0)  # push strength in units of the patch's norm at each position (dose_v1)
MIN_SHIFT = (
    0.10  # suite_split's MIN_FRAC: below this the whole patch barely moves the answer, so effect ÷ size is noise
)
BLUE, ORANGE, DARK = "#2a78d6", "#eb6834", "#52514e"  # dataviz palette slots 1-2 (CVD-checked) and neutral ink
MODEL_NAME = pins.MODEL_REPO.split("/")[-1]


def rows(where: str, run: str | None, file: str) -> list[dict]:
    """A results file of Zach's as dicts; [] if the run is not configured or not in results/."""
    p = pins.RESULTS_DIR / where / str(run) / file
    return [json.loads(line) for line in p.read_text().splitlines()] if run and p.exists() else []


def summary(where: str, run: str | None) -> dict:
    p = pins.RESULTS_DIR / where / str(run) / "summary.json"
    return json.loads(p.read_text()) if run and p.exists() else {}


def table(header: list[str], body: list[list]) -> None:
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(x) for x in r) + " |" for r in body]
    display(Markdown("\n".join(lines)))


def grid(n: int, height: float = 2.3, sharey: bool = True):
    """Small multiples, one panel per family, in the repo's figure style."""
    cols = min(n, 3)
    nrows = -(-n // cols)
    fig, axes = plt.subplots(
        nrows, cols, figsize=(3.8 * cols, height * nrows + 1.6), sharey=sharey, squeeze=False, facecolor=SURFACE
    )
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    for ax in axes.flat[:n]:
        style_axes(ax)
    return fig, list(axes.flat[:n])


def show(fig, title: str, note: str, handles: list) -> None:
    """Title and note above the panels, legend below (bridge_jspace.plotting.finish, shown instead of saved)."""
    h, w = fig.get_figheight(), fig.get_figwidth()
    note_lines = textwrap.wrap(note, width=int(w * 15.5))
    head = 0.42 + 0.17 * len(note_lines)
    fig.text(0.01, 1 - 0.1 / h, title, ha="left", va="top", fontsize=12, color=INK)
    fig.text(0.01, 1 - 0.36 / h, "\n".join(note_lines), ha="left", va="top", fontsize=8.5, color=MUTED, linespacing=1.4)
    fig.legend(handles=handles, loc="lower center", ncols=len(handles), frameon=False, fontsize=9, labelcolor=INK)
    fig.tight_layout(rect=(0, 0.45 / h, 1, 1 - head / h))
    plt.show()


def title(ax, f: str, extra: str = "") -> None:
    ax.set_title(LABEL[f] + extra, fontsize=9.5, color=INK, loc="left")


def agreement(mine: list[dict], theirs: dict, key, by=None) -> None:
    """How many of my edited runs Zach's run has (same item, layer, edit), how far their shifts are apart, and how
    often the top token's role (A_alt / A_org / other) agrees: per family, and with `by` also per kind of edit."""
    groups = [("family", lambda o: LABEL[o["family"]])] + ([("edit", by)] if by else [])
    for name, group in groups:
        cells: dict[str, list] = collections.defaultdict(list)
        for o in mine:
            if key(o) in theirs:
                cells[group(o)].append((o, theirs[key(o)]))
        body = []
        for g, pairs in cells.items():
            d = [abs(o["frac"] - z["frac"]) for o, z in pairs]
            same = statistics.mean(o["role"] == z["role"] for o, z in pairs)
            body.append([g, len(pairs), f"{statistics.median(d):.3f}", f"{max(d):.3f}", f"{same:.0%}"])
        table([name, "edited runs matched", "median abs Δshift", "max abs Δshift", "same top token"], body)

## 1. A random scene

The scene is drawn from the 150 scenes of `map_A8_v1` (every later run of Zach's uses them), restricted to the scenes that are also in his read/write and knock-out runs, so that every check can be compared. Only the tokenizer is needed here; the model loads in §2.

In [ ]:
zach = {
    "competence": rows("suite_v1", R["competence"], "items.jsonl"),
    "map": rows("suite_split", R["map"], "outcomes.jsonl"),
    "map_cases": rows("suite_split", R["map"], "cases.jsonl"),
    "parts": rows("suite_split", R["parts"], "outcomes.jsonl"),
    "reads": rows("read_write", R["read_write"], "reads.jsonl"),
    "rw": rows("read_write", R["read_write"], "outcomes.jsonl"),
    "rw_early": rows("read_write", R["read_write_early"], "outcomes.jsonl"),
    "ablate": rows("ablate_pilot", R["ablate"], "rows.jsonl"),
    "country_map": rows("control_split", R["country_map"], "outcomes.jsonl"),
    "country_cases": rows("control_split", R["country_map"], "cases.jsonl"),
}
print({k: len(v) for k, v in zach.items()})


def run_order(rs: list[dict]) -> dict[str, int]:
    """Each case's index in a run, by its first row: the k that read_write seeds its random directions with."""
    order: dict[str, int] = {}
    for r in rs:
        order.setdefault(r["name"], len(order))
    return order


K_RW, K_RW_EARLY = run_order(zach["rw"]), run_order(zach["rw_early"])
K_PARTS = {c["id"]: k for k, c in enumerate(zach["map_cases"])}  # run_parts counts every map case, windowless too
in_ablate = {r["id"] for r in zach["ablate"]}
id_of = {(c["family"], c["scene"]): c["id"] for c in zach["map_cases"] if c["family"] in CORE}  # NAMES/CODE: no scene
scenes = sorted({s for (_, s) in id_of if all((f, s) in id_of for f in CORE)})
covered = [s for s in scenes if all(id_of[f, s] in K_RW and (not in_ablate or id_of[f, s] in in_ablate) for f in CORE)]
rng = random.Random(SEED)
scene = rng.choice(covered or scenes)
country = {}
for f in COUNTRIES:
    pool = [
        c["id"]
        for c in zach["country_cases"]
        if c["family"] == f and c["id"] in K_RW and (not in_ablate or c["id"] in in_ablate)
    ]
    country[f] = rng.choice(pool)
print(
    f"scene {scene}: seed {SEED}, one of the {len(covered)} map scenes all of Zach's runs here cover ({len(scenes)} in the map)"
)
print("country items:", country)

In [ ]:
tok = S.tokenizer()
cases: dict[str, dict] = {}
for f in CORE:
    item = next(it for it in S.load(f) if it["id"] == id_of[f, scene])
    cases[f] = {**item, **P.prepare(item, tok)}
bind = E.ITEM_SETS["bind_recall"][2]()
for f, uid in country.items():  # as read_write.country_cases
    index = int(uid.split("-")[1])  # exp09's uid: family and index in data/bind_recall.json
    item = bind[index]
    assert E.uid(item, index) == uid
    c = {**item, **E.prepare(item, index, tok, "chat_prefill"), "id": uid, "scene": uid}
    c |= {"cf_bridge": item["target"], "cf_answer": item["target_answer"], "answer_kind": "word"}
    c["read_ids"] = c["ids"]
    c["tok_A"], c["tok_b"] = (S.answer_token(tok, a, "word") for a in (item["answer"], item["target_answer"]))
    cases[f] = c


def changes(c: dict, source: str) -> str:
    """What the swap ("cf") or control ("off") prompt changes: each run of changed tokens, clean → edited."""
    a, b, pos = c["ids"]["clean"], c["ids"][source], c["changed"][source]
    runs = [[pos[0]]]
    for i in pos[1:]:
        (runs[-1].append(i) if i == runs[-1][-1] + 1 else runs.append([i]))
    return ", ".join(
        f"{tok.decode([a[i] for i in r]).strip()!r} → {tok.decode([b[i] for i in r]).strip()!r}" for r in runs
    )


def steps(f: str, c: dict) -> str:
    if f in CORE:
        return (
            f"{c['hop1']['question']} → **{c['hop1']['answer']}**; {c['hop2']['question']} → **{c['hop2']['answer']}**"
        )
    where = f"the country where {c['cue']} " + ("is located" if f == "R" else "lives")
    return f"{where} → **{c['bridge']}**; its capital → **{c['answer']}**"


print("Context of the four animal families:\n ", cases["BL"]["context"])
for f in COUNTRIES:
    print(f"Prompt of {LABEL[f]}:\n ", cases[f]["prompt"])
table(
    ["family", "question (clean)", "the two steps alone", "B_org → B_alt", "A_org → A_alt", "swap changes",
     "control changes"],
    [
        [LABEL[f], c.get("question", "(the prompt's last sentence)"), steps(f, c), f"{c['bridge']} → {c['cf_bridge']}",
         f"{c['answer']} → {c['cf_answer']}", changes(c, "cf"), changes(c, "off")]
        for f, c in cases.items()
    ],
)  # fmt: skip

This is the prompt as the model sees it (chat template, thinking off, "Answer:" prefilled). The tokens in ⟦brackets⟧ are the **question tokens**: the question after the swap's changed tokens, to the end of the prompt. §4–§7 patch and push residuals there.

In [ ]:
def marked(c: dict, group: str = "question") -> str:
    pos = set(c["groups"][group])
    return "".join(f"⟦{tok.decode([t])}⟧" if i in pos else tok.decode([t]) for i, t in enumerate(c["ids"]["clean"]))


print(marked(cases["RL"]))
for f in COUNTRIES[:1]:
    print("\n" + marked(cases[f]))

## 2. The model, and competence

Before editing anything: the model has to answer the clean prompt with A_org, the swap prompt with A_alt, the control prompt with A_org, and each step on its own; otherwise there is nothing to compare an edit with. `p(answer)` is the probability of the expected answer's first token where Zach's scripts read it (after "Answer:"; for a digit, with a space appended).

In [ ]:
from bridge_jspace.lens import JLens
from bridge_jspace.model import load_model

t0 = time.time()
model = load_model()
lens = JLens(model, pins.FROZEN_LENS)  # the frozen J++, sha256-checked (the first check hashes 3.3 GB)
print(f"loaded in {time.time() - t0:.0f}s; GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
def ids_t(ids: list[int]) -> torch.Tensor:
    return torch.tensor([ids], device="cuda")


def answer_p(ids: list[int], answer: str, kind: str, edits: dict | None = None) -> float:
    """p(the answer's first token) where suite_v1 reads it: after 'Answer:', a digit answer's space appended."""
    logits, _ = model.run(ids_t(ids + (S.ids(tok, " ") if kind == "digit" else [])), edits)
    return float(logits.softmax(-1)[S.answer_token(tok, answer, kind)])


def greedy(ids: list[int], edits: dict | None = None) -> str:
    return model.generate(ids_t(ids), 8, edits)[1].strip().replace("\n", " ")


def hop_ids(c: dict, hop: dict) -> list[int]:
    return S.ids(tok, S.render(tok, hop.get("context", c["context"]), hop["question"]))


PROMPT = {"clean": "clean", "cf": "swap", "off": "control", "hop1": "step 1 alone", "hop2": "step 2 alone",
          "hop3": "step 3 alone"}  # fmt: skip
comp_z = {(r["family"], r["id"], r["kind"]): r for r in zach["competence"]}
body = []
for f, c in cases.items():
    k = c["answer_kind"]
    prompts = [("clean", c["ids"]["clean"], c["answer"], k), ("cf", c["ids"]["cf"], c["cf_answer"], k)]
    prompts += [("off", c["ids"]["off"], c["answer"], k)]
    prompts += [(h, hop_ids(c, c[h]), c[h]["answer"], c[h]["kind"]) for h in ("hop1", "hop2", "hop3") if h in c]
    for name, ids, answer, kind in prompts:
        z = comp_z.get((f, c["id"], name))
        p = answer_p(ids, answer, kind)
        body.append(
            [LABEL[f], PROMPT[name], answer, repr(greedy(ids)), f"{p:.3f}", f"{z['p_answer']:.3f}" if z else "-"]
        )
table(["family", "prompt", "expected", "greedy, 8 tokens", "p(answer)", "Zach's p (competence run)"], body)

## 3. Lens read: does J++ see the bridge?

J++ is a map J_l for each layer, from the residual at layer l to the final block's; the readout is `unembed(h @ J_lᵀ)`. At each question token we take the bridge word's rank among word tokens (formatting tokens removed), then the best rank over the question tokens. "Read" in Zach's runs means top 10. B_alt is not in the clean prompt, so its rank is the floor: what chance gives.

Expected (row 32, n=60), share read: memory → prompt 95%, memory → memory 88%, prompt → memory 63%, prompt → prompt 7%, countries 100%.

In [ ]:
LAYERS = sorted(set(P.MAP_LAYERS) | set(R["window"]) | set(R["early"]))
h = {  # block outputs [seq, d] at LAYERS for the clean, swap and control prompts: everything below patches these
    f: {s: model.run(ids_t(c["ids"][s]), record=tuple(LAYERS))[1] for s in ("clean", "cf", "off")}
    for f, c in cases.items()
}


def ranks(logits: torch.Tensor, word: str) -> list[int]:
    """The word's rank among word tokens at each row (position) of a lens readout."""
    ids = model.surface_ids(word)
    return [model.word_rank(row, ids) for row in logits] if ids else []


reads = {}
for f, c in cases.items():
    pos = c["groups"]["question"]
    for la in LAYERS:
        logits = lens.readout(h[f]["clean"][la][pos], la)
        reads[f, la] = (min(ranks(logits, c["bridge"]), default=None), min(ranks(logits, c["cf_bridge"]), default=None))

fig, axes = grid(len(FAMILIES))
worst = max(r for v in reads.values() for r in v if r)
for ax, f in zip(axes, FAMILIES, strict=True):
    shade_band(ax, min(R["window"]), max(R["window"]), label=False)
    ax.plot(LAYERS, [reads[f, la][1] or float("nan") for la in LAYERS], color=MUTED, lw=2)
    ax.plot(LAYERS, [reads[f, la][0] or float("nan") for la in LAYERS], color=BLUE, lw=2, marker="o", ms=4)
    ax.axhline(10, color=DARK, lw=0.8, ls=":")
    ax.set_yscale("log")
    ax.set_ylim(worst * 1.5, 0.8)
    ax.set_xlabel("layer", fontsize=8.5, color=MUTED)
    title(ax, f)
axes[0].set_ylabel("best rank over question tokens", fontsize=8.5, color=MUTED)
show(
    fig,
    "Does the J++ lens see the bridge at the question tokens?",
    f"{MODEL_NAME}, scene {scene} and countries {list(country.values())}, one item per family. Rank 1 = the lens's top "
    f"word; dotted line = top 10 (Zach's 'read'). Shaded: the carry window, layers {R['window']}.",
    [Line2D([], [], color=BLUE, lw=2, marker="o", label="B_org, the bridge"),
     Line2D([], [], color=MUTED, lw=2, label="B_alt (not in this prompt): the floor")],
)  # fmt: skip

In [ ]:
z_reads = {(r["name"], r["layer"]): (r["rank_bridge"], r["rank_x"]) for r in zach["reads"]}
table(
    ["family", "layer", "B_org rank (here)", "B_org rank (Zach)", "B_alt rank (here)", "B_alt rank (Zach)"],
    [
        [LABEL[f], la, reads[f, la][0], z_reads.get((cases[f]["id"], la), ("-",))[0], reads[f, la][1],
         z_reads.get((cases[f]["id"], la), ("-", "-"))[1]]
        for f in FAMILIES for la in R["window"]
    ],
)  # fmt: skip

What the lens reads: J++'s top 10 words at the question token where the bridge ranks highest (the carry window's middle layer). The bridge in **bold**.

In [ ]:
la = R["window"][len(R["window"]) // 2]
body = []
for f, c in cases.items():
    pos = c["groups"]["question"]
    logits = lens.readout(h[f]["clean"][la][pos], la)
    r = ranks(logits, c["bridge"])
    best = min(range(len(r)), key=r.__getitem__)
    top = logits[best].masked_fill(model.formatting_mask, float("-inf")).topk(10).indices.tolist()
    bridge_ids = set(model.surface_ids(c["bridge"]))
    words = [f"**{model.decoded[t].strip()}**" if t in bridge_ids else model.decoded[t].strip() for t in top]
    body.append([LABEL[f], la, repr(tok.decode([c["ids"]["clean"][pos[best]]])), r[best], " · ".join(words)])
table(["family", "layer", "at token", "B_org rank", "lens top 10"], body)

## 4. Patch map: where does the model carry the bridge?

Activation patching. Run the swap prompt (bridge = B_alt), take its residual at layer l at some positions, and add the difference `h_swap − h_clean` to the clean run there. If the model carries the bridge at those positions and that layer, the answer moves toward A_alt.

The measure (Zach, 2026-10-19), at the answer's first token: `gap = logit(A_alt) − logit(A_org)`, and **shift** `= (gap − gap_clean) / (gap_swap − gap_clean)`: 0% = as clean, 100% = as the swap prompt. A **flip** = the top token becomes A_alt. The control patch (from the control prompt) should not move the answer.

First by hand: one patch per family at the carry window's middle layer, then the greedy reply for the family it moves most.

In [ ]:
for c in cases.values():
    P.unedited_ld(model, c)  # gap of the unedited clean and swap prompts: the two ends of the shift

la = R["window"][len(R["window"]) // 2]
body, by_hand = [], {}
for f, c in cases.items():
    pos = c["groups"]["question"]
    patch = (
        h[f]["cf"][la][pos] - h[f]["clean"][la][pos]
    )  # [question tokens, d]: what the swap run has there that clean lacks
    logits, _ = model.run(ids_t(c["read_ids"]["clean"]), {la: AddRows(patch, pos)})
    gap = float(logits[c["tok_b"]] - logits[c["tok_A"]])
    by_hand[f] = (gap - c["ld_clean"]) / (c["ld_cf"] - c["ld_clean"])
    body.append([LABEL[f], len(pos), f"{float(patch.norm(dim=-1).mean()):.0f}", f"{c['ld_clean']:+.1f}",
                 f"{c['ld_cf']:+.1f}", f"{gap:+.1f}", f"{by_hand[f]:.0%}", repr(model.decoded[int(logits.argmax())])])  # fmt: skip
table(
    ["family", "question tokens", "patch norm per token", "gap clean", "gap swap", "gap patched", "shift", "top token"],
    body,
)

f = max(by_hand, key=by_hand.get)
c, pos = cases[f], cases[f]["groups"]["question"]
patch = h[f]["cf"][la][pos] - h[f]["clean"][la][pos]
control = h[f]["off"][la][pos] - h[f]["clean"][la][pos]
print(f"{LABEL[f]}, layer {la} (A_org {c['answer']!r}, A_alt {c['cf_answer']!r})")
print("  greedy, clean:              ", repr(greedy(c["ids"]["clean"])))
print("  greedy, with the swap patch:", repr(greedy(c["ids"]["clean"], {la: AddRows(patch, pos)})))
print("  greedy, with the control patch:", repr(greedy(c["ids"]["clean"], {la: AddRows(control, pos)})))

Now the full map, as `suite_split run_map` makes it: every 4th layer, at the question tokens and at the last token alone, the swap patch and the control patch.

In [ ]:
patch_out: list[dict] = []
for f, c in cases.items():
    jobs = []
    for group in ("question", "last"):
        pos = c["groups"][group]
        for la in P.MAP_LAYERS:
            for source in ("cf", "off"):
                patch = h[f][source][la][pos] - h[f]["clean"][la][pos]
                jobs.append(({"group": group, "layer": la, "source": source}, {la: AddRows(patch, pos)}))
    P.read_jobs(model, c, jobs, patch_out)


def shift_at(out: list[dict], f: str, **where) -> float:
    v = [o["frac"] for o in out if o["family"] == f and all(o.get(k) == x for k, x in where.items())]
    return statistics.mean(v) if v else float("nan")


fig, axes = grid(len(FAMILIES))
for ax, f in zip(axes, FAMILIES, strict=True):
    shade_band(ax, min(R["window"]), max(R["window"]), label=False)
    for group, source, color in (("question", "off", MUTED), ("last", "cf", ORANGE), ("question", "cf", BLUE)):
        ys = [shift_at(patch_out, f, group=group, layer=la, source=source) for la in P.MAP_LAYERS]
        ax.plot(P.MAP_LAYERS, ys, color=color, lw=2, marker="o" if color == BLUE else None, ms=4)
    ax.axhline(0, color=DARK, lw=0.8)
    percent_axis(ax)
    ax.set_xlabel("layer", fontsize=8.5, color=MUTED)
    title(ax, f)
axes[0].set_ylabel("shift toward A_alt", fontsize=8.5, color=MUTED)
show(
    fig,
    "Where does the model carry the bridge? Patch the swap prompt's residual into the clean run, one layer at a time",
    f"{MODEL_NAME}, one item per family (scene {scene}). 0% = the clean prompt's answer A_org, 100% = the swap prompt's A_alt. "
    f"Shaded: the carry window, layers {R['window']}, below the layer where the last token alone holds the answer.",
    [Line2D([], [], color=BLUE, lw=2, marker="o", label="swap patch, question tokens"),
     Line2D([], [], color=ORANGE, lw=2, label="swap patch, last token only"),
     Line2D([], [], color=MUTED, lw=2, label="control patch, question tokens")],
)  # fmt: skip

z_map = {(r["name"], r["group"], r["layer"], r["source"]): r for r in zach["map"] + zach["country_map"]}
agreement(patch_out, z_map, lambda o: (o["name"], o["group"], o["layer"], o["source"]))

## 5. Patch split: which part of the patch moves the answer?

At each layer of the carry window, the same patch is split into parts, and each part is added on its own:
- **J-space part**: a 25-atom signed OMP over the unit J++ vectors of all letter tokens (`v_t = (g ⊙ W_U[t]) @ J_l`). **Rest** = patch − J-space part.
- **Bridge-word part**: the projection on the lens vectors of B_org and B_alt themselves. **J-space minus bridge words**: the OMP of patch − bridge-word part.
- Nulls: each part along a random direction at the same norms; a *matched random piece* (the same norm and cosine with the patch); the *random-dictionary part*, an OMP over a random dictionary of the same size, which says how much effect ÷ size the fitting procedure alone gives (row 31: 0.15–0.4).

**Effect ÷ size** (Zach's "concentration") = shift(part) / (share × shift(whole patch)), with share = |part|² / |patch|²: 1 means the part does exactly what its size predicts. It is shown only where the whole patch shifts the answer at least 10%. Rows 27–31 (n=150): prompt → prompt 0.36, memory → prompt 0.27, prompt → memory 1.36, memory → memory 1.60; countries 2.86 and 2.47, about 90% of it the bridge words.

In [ ]:
parts_manifest = json.loads((P.RESULTS / R["parts"] / "manifest.json").read_text())
window = {f: parts_manifest["map"]["parts_window"][f] for f in CORE} | {f: list(R["country_window"]) for f in COUNTRIES}
letters = model.letter_token_ids()
d_model = h["BL"]["clean"][LAYERS[0]].shape[1]
print(
    f"J-space dictionary: {len(letters):,} letter tokens x d={d_model}, {len(letters) * d_model * 4 / 1e9:.1f} GB a layer"
)
print("carry windows:", {LABEL[f]: w for f, w in window.items()})

split_out: list[dict] = []
share: dict[tuple, float] = {}
split_jobs: dict[str, list] = {f: [] for f in cases}
for la in sorted({la for w in window.values() for la in w}):
    jdict = lens.vectors(la, letters)  # [letters, d] unit J++ vectors: the J-space dictionary at this layer
    g = torch.Generator(device="cuda").manual_seed(777 + la)  # suite_split's random dictionary, same seed
    rdict = torch.nn.functional.normalize(torch.randn(jdict.shape, device="cuda", generator=g), dim=-1)
    for i, (f, c) in enumerate(cases.items()):
        if la not in window[f]:
            continue
        k = K_PARTS.get(c["id"], 10**5 + i)  # run_parts' case index seeds its nulls; countries: our own seeds
        pos = c["groups"]["question"]
        patch = h[f]["cf"][la][pos] - h[f]["clean"][la][pos]
        control = h[f]["off"][la][pos] - h[f]["clean"][la][pos]
        pair = torch.tensor([model.concept_token(c["bridge"]), model.concept_token(c["cf_bridge"])], device="cuda")
        words = lens2_part(patch, lens.vectors(la, pair))
        jspace = signed_omp(jdict, patch, K_ATOMS)
        # Zach's part names (the keys his outcomes use): lens2 = bridge words, jspace_wo_own = J-space minus bridge words
        parts = {"lens2": words, "not_lens2": patch - words, "jspace": jspace, "not_jspace": patch - jspace,
                 "jspace_wo_own": signed_omp(jdict, patch - words, K_ATOMS), "rand_omp": signed_omp(rdict, patch, K_ATOMS)}  # fmt: skip
        for name in ("lens2", "jspace", "jspace_wo_own", "rand_omp"):
            share[f, la, name] = float(parts[name].norm() ** 2 / patch.norm() ** 2)
        jobs = split_jobs[f]
        jobs.append(({"layer": la, "part": "full", "source": "cf", "null": False}, {la: AddRows(patch, pos)}))
        jobs.append(({"layer": la, "part": "full", "source": "off", "null": False}, {la: AddRows(control, pos)}))
        for pi, (name, part) in enumerate(parts.items()):  # as stage_run_parts, seeds included
            jobs.append(({"layer": la, "part": name, "source": "cf", "null": False}, {la: AddRows(part, pos)}))
            noise = null_rows(part, torch.Generator(device="cuda").manual_seed(P.NULL_SEED + 100000 * k + 10 * la + pi))
            jobs.append(({"layer": la, "part": name, "source": "cf", "null": True}, {la: AddRows(noise, pos)}))
        for mi, name in enumerate(("lens2", "jspace")):
            gen = torch.Generator(device="cuda").manual_seed(P.NULL_SEED + 100000 * k + 10 * la + 5 + mi)
            matched = P.matched_rows(parts[name], patch, gen)
            jobs.append(({"layer": la, "part": f"{name}_matched", "source": "cf", "null": False}, {la: AddRows(matched, pos)}))  # fmt: skip
    del jdict, rdict
    torch.cuda.empty_cache()
for f, c in cases.items():
    P.read_jobs(model, c, split_jobs[f], split_out)


def effect_per_size(f: str, part: str) -> str:
    """This item's effect ÷ size for a part: its shift over (its share of the patch x the whole patch's shift)."""
    whole = shift_at(split_out, f, part="full", source="cf")
    if whole < MIN_SHIFT:
        return f"n/a (patch {whole:.0%})"
    s = statistics.mean(share[f, la, part] for la in window[f])
    return f"{shift_at(split_out, f, part=part, null=False, source='cf') / (s * whole):.2f}"


z_parts_summary = summary("suite_split", R["parts"]).get("families", {})
body = []
for f in FAMILIES:
    z = z_parts_summary.get(f, {})
    zc = z.get("frac", {}).get("jspace", {}).get("concentration")
    zr = z.get("calibration", {}).get("rand_omp", {}).get("concentration")
    body.append([LABEL[f], f"{shift_at(split_out, f, part='full', source='cf'):.0%}",
                 f"{shift_at(split_out, f, part='jspace', null=False):.1%}",
                 f"{statistics.mean(share[f, la, 'jspace'] for la in window[f]):.1%}", effect_per_size(f, "jspace"),
                 f"{zc[0]:.2f} [{zc[1]:.2f}, {zc[2]:.2f}]" if zc else "-", effect_per_size(f, "rand_omp"),
                 f"{zr[0]:.2f}" if zr else "-"])  # fmt: skip
table(
    ["family", "whole patch: shift", "J-space part: shift", "J-space part: share of the patch",
     "effect ÷ size (this item)", "Zach, family [95% CI]", "random-dictionary part: effect ÷ size", "Zach, family"],
    body,
)  # fmt: skip

In [ ]:
SHOWN = [  # (Zach's part key, null, label, color)
    ("full", False, "whole patch", DARK),
    ("not_jspace", False, "rest (not J-space)", DARK),
    ("jspace", False, "J-space part", BLUE),
    ("lens2", False, "bridge-word part", BLUE),
    ("jspace_wo_own", False, "J-space minus bridge words", BLUE),
    ("rand_omp", False, "random-dictionary part (null)", MUTED),
    ("jspace_matched", False, "matched random piece (null)", MUTED),
    ("jspace", True, "J-space part, random direction (null)", MUTED),
]
fig, axes = grid(len(FAMILIES), height=2.6)
for ax, f in zip(axes, FAMILIES, strict=True):
    vals = [shift_at(split_out, f, part=p, null=n, source="cf") for p, n, _, _ in SHOWN]
    ax.barh(range(len(SHOWN)), vals, color=[col for *_, col in SHOWN], height=0.7, edgecolor=SURFACE, linewidth=2)
    for y, v in enumerate(vals):
        ax.annotate(
            f"{v:.0%}", (max(v, 0), y), xytext=(3, 0), textcoords="offset points", va="center", fontsize=7.5, color=INK
        )
    ax.set_yticks(range(len(SHOWN)), [lab for _, _, lab, _ in SHOWN], fontsize=7.5)
    ax.invert_yaxis()
    ax.axvline(0, color=DARK, lw=0.8)
    percent_axis(ax, "x")
    title(ax, f)
show(
    fig,
    "Which part of the patch moves the answer? Each part of the swap patch added on its own",
    f"{MODEL_NAME}, one item per family; shift toward A_alt, averaged over the carry window's layers. "
    "A part that carries the bridge moves the answer more than its size predicts (effect ÷ size > 1, table above).",
    [Patch(color=DARK, label="whole patch / rest"), Patch(color=BLUE, label="J-space parts"), Patch(color=MUTED, label="nulls")],
)  # fmt: skip

z_parts = {(r["name"], r["layer"], r["part"], r["null"], r["source"]): r for r in zach["parts"]}
agreement(split_out, z_parts, lambda o: (o["name"], o["layer"], o["part"], o["null"], o["source"]))

## 6. Read vs write: can a push along the lens direction write the bridge?

The **lens direction** points from B_org's unit lens vector to B_alt's (`v(B_alt) − v(B_org)`). At each question token it is scaled to the swap patch's norm there, times a strength (0.5–16×), and added. If the lens reads the bridge in the form the model uses, the push should do what the swap patch does. Control: a **random direction** at the same norms. "All window layers" = the push at every carry-window layer at once.

Expected. Row 37 (n=80): in the carry window, memory → prompt is not written at any strength (≤3% flips), memory → memory is (36% shift at 4×), countries flip 95–98%. Row 52 revises the headline: at early layers 16–28, where the patch does nothing yet, a strong 8–16× push "writes" memory → prompt too (35–61%), and even prompt → prompt. Zach reads this as injecting the word into the question rather than writing the bridge (untested); testing it is item 4 of "Dan: depth".

In [ ]:
def write_jobs(c: dict, f: str, layers, k: int, band: bool) -> list:
    """read_write's edits at `layers`: the swap patch, the lens direction and a random one at each strength (its seeds).
    Part keys stay Zach's: "copy" = the swap patch, "lens_x4" = lens direction at 4x, "random_x4" = random at 4x."""
    pos = c["groups"]["question"]
    pair = torch.tensor([model.concept_token(c["bridge"]), model.concept_token(c["cf_bridge"])], device="cuda")
    jobs, by_part = [], collections.defaultdict(dict)
    for la in layers:
        patch = h[f]["cf"][la][pos] - h[f]["clean"][la][pos]
        norms = patch.norm(dim=-1, keepdim=True)
        v = lens.vectors(la, pair)
        direction = torch.nn.functional.normalize(v[1] - v[0], dim=-1).expand_as(patch)  # v(B_alt) - v(B_org)
        gen = torch.Generator(device="cuda").manual_seed(10_000 * k + la)
        edits = {"copy": patch}
        for sc in SCALES:
            edits[f"lens_x{sc:g}"] = sc * norms * direction
            edits[f"random_x{sc:g}"] = null_rows(sc * norms * direction, gen)
        for name, vec in edits.items():
            jobs.append(({"layer": la, "part": name, "source": "cf", "null": False}, {la: AddRows(vec, pos)}))
            by_part[name][la] = AddRows(vec, pos)
    if band:
        jobs += [
            ({"layer": "band", "part": name, "source": "cf", "null": False}, dict(e)) for name, e in by_part.items()
        ]
    return jobs


rw_out: list[dict] = []
rw_early_out: list[dict] = []
for i, (f, c) in enumerate(cases.items()):
    P.read_jobs(model, c, write_jobs(c, f, R["window"], K_RW.get(c["id"], 10**5 + i), band=True), rw_out)
    if WRITE_EARLY and R["early"]:
        P.read_jobs(
            model, c, write_jobs(c, f, R["early"], K_RW_EARLY.get(c["id"], 10**5 + i), band=False), rw_early_out
        )


def flips_at(out: list[dict], f: str, **where) -> float:
    v = [o["role"] == "b" for o in out if o["family"] == f and all(o.get(k) == x for k, x in where.items())]
    return statistics.mean(v) if v else float("nan")


def plot_write(out: list[dict], layers, label: str) -> None:
    fig, axes = grid(len(FAMILIES))
    single = [o for o in out if o["layer"] != "band"]
    for ax, f in zip(axes, FAMILIES, strict=True):
        for kind, color in (("random", MUTED), ("lens", BLUE)):
            ys = [shift_at(single, f, part=f"{kind}_x{sc:g}") for sc in SCALES]
            ax.plot(SCALES, ys, color=color, lw=2, marker="o" if kind == "lens" else None, ms=4)
        ax.axhline(shift_at(single, f, part="copy"), color=DARK, lw=1.2, ls="--")
        ax.axhline(0, color=DARK, lw=0.8)
        ax.set_xscale("log", base=2)
        ax.set_xticks(SCALES, [f"{sc:g}×" for sc in SCALES], fontsize=8)
        percent_axis(ax)
        ax.set_xlabel("push strength, × the patch's norm", fontsize=8.5, color=MUTED)
        title(ax, f, f"   (read: rank {min(reads[f, la][0] or 10**9 for la in layers)})")
    axes[0].set_ylabel("shift toward A_alt", fontsize=8.5, color=MUTED)
    show(
        fig,
        f"Can the lens direction write the bridge? Push at the question tokens, {label}",
        f"{MODEL_NAME}, one item per family; shift averaged over single-layer pushes at layers {list(layers)}. "
        "'read: rank' = the bridge's best lens rank at those layers (§3).",
        [Line2D([], [], color=BLUE, lw=2, marker="o", label="lens direction v(B_alt) − v(B_org)"),
         Line2D([], [], color=MUTED, lw=2, label="random direction, same norms"),
         Line2D([], [], color=DARK, lw=1.2, ls="--", label="the swap patch itself")],
    )  # fmt: skip


plot_write(rw_out, R["window"], "in the carry window")
if rw_early_out:
    plot_write(rw_early_out, R["early"], "at early layers")

In [ ]:
single = [o for o in rw_out if o["layer"] != "band"]
table(
    ["family", "swap patch"] + [f"lens {sc:g}×" for sc in SCALES] + ["lens 4×, all window layers"],
    [
        [LABEL[f], f"{flips_at(single, f, part='copy'):.0%}"]
        + [f"{flips_at(single, f, part=f'lens_x{sc:g}'):.0%}" for sc in SCALES]
        + [f"{flips_at(rw_out, f, layer='band', part='lens_x4'):.0%}"]
        for f in FAMILIES
    ],
)  # flip rates, single layers averaged over the carry window


def edit_kind(o: dict) -> str:
    kind, _, scale = o["part"].partition("_x")
    where = "all window layers" if o["layer"] == "band" else "early" if o["layer"] in R["early"] else "window"
    strength = "" if not scale else " ≤4×" if float(scale) <= 4 else " 8–16×"
    return f"{ {'copy': 'swap patch', 'lens': 'lens', 'random': 'random'}[kind] }{strength}, {where}"


# both runs have "band" rows (all of their own layers at once): the early run's must not stand in for the window's
z_rw = {(r["name"], r["layer"], r["part"]): r for r in zach["rw_early"] if r["layer"] != "band"}
z_rw |= {(r["name"], r["layer"], r["part"]): r for r in zach["rw"]}
agreement(rw_out + rw_early_out, z_rw, lambda o: (o["name"], o["layer"], o["part"]), by=edit_kind)

## 7. Knock-out: does the task need the J-space?

The original paper's test. At the carry-window layers (all at once), at **every** position, subtract the residual's J-space part: a 25-atom OMP over J++'s letter-token dictionary. Controls:
- **same-size rotated removal**: the same J-space part turned by a fixed random rotation, so the edit has the same size (row 55);
- **random-dictionary removal**: an OMP over a random dictionary (row 42).

We look at p(answer) for the whole question and for its last step alone. In prompt → prompt and memory → prompt the last step is a lookup in the prompt; in prompt → memory and memory → memory, a recall from memory.

Expected. Row 42 (n=100, accuracy): families ending in a prompt lookup 99%, ending in a recall 19–22%, countries 26–34%, controls 98–100%. Confound (row 53): word answers lose ~35% even in a pure lookup, so part of the countries' drop is their answer type.

In [ ]:
def knock_prompts(f: str, c: dict) -> list[tuple[str, list[int], int]]:
    """ablate_pilot's prompts: the whole question, and for the animal families its last step alone."""
    out = [("whole question", c["read_ids"]["clean"], c["tok_A"])]
    if f in CORE:
        hop = c.get("hop3") or c["hop2"]
        ids = hop_ids(c, hop) + (S.ids(tok, " ") if hop["kind"] == "digit" else [])
        out.append(("last step alone", ids, S.answer_token(tok, hop["answer"], hop["kind"])))
    return out


knock: dict[tuple, tuple[bool, float]] = {}


def run_condition(name: str, edits: dict) -> None:
    for f, c in cases.items():
        for kind, ids, t in knock_prompts(f, c):
            logits, _ = model.run(ids_t(ids), edits)
            knock[f, kind, name] = (int(logits.argmax()) == t, float(logits.log_softmax(-1)[t]))


layers = R["window"]
t0 = time.time()
run_condition("nothing removed", {})
jdicts = {la: lens.vectors(la, letters) for la in layers}
run_condition("J-space removed", {la: AB.Remove(jdicts[la]) for la in layers})
run_condition(
    "same-size rotated removal", {la: AB.Rotated(jdicts[la], AB.rotation(d_model, 888 + la)) for la in layers}
)
shape = jdicts[layers[0]].shape
del jdicts
torch.cuda.empty_cache()
rdicts = {}
for la in layers:  # ablate_pilot's random dictionary: same size, seeded per layer
    g = torch.Generator(device="cuda").manual_seed(777 + la)
    rdicts[la] = torch.nn.functional.normalize(torch.randn(shape, device="cuda", generator=g), dim=-1)
run_condition("random-dictionary removal", {la: AB.Remove(rdicts[la]) for la in layers})
del rdicts
torch.cuda.empty_cache()
print(f"{time.time() - t0:.0f}s")

In [ ]:
CONDS = [("nothing removed", DARK), ("J-space removed", BLUE), ("same-size rotated removal", ORANGE),
         ("random-dictionary removal", MUTED)]  # fmt: skip
fig, axes = grid(len(FAMILIES), height=2.4)
for ax, f in zip(axes, FAMILIES, strict=True):
    kinds = [k for k in ("whole question", "last step alone") if (f, k, "nothing removed") in knock]
    for j, (cond, color) in enumerate(CONDS):
        xs = [i + (j - 1.5) * 0.2 for i in range(len(kinds))]
        ax.bar(
            xs, [math.exp(knock[f, k, cond][1]) for k in kinds], width=0.2, color=color, edgecolor=SURFACE, linewidth=2
        )
    ax.set_xticks(range(len(kinds)), kinds, fontsize=8)
    ax.set_ylim(0, 1.05)
    percent_axis(ax)
    title(ax, f)
axes[0].set_ylabel("p(answer)", fontsize=8.5, color=MUTED)
show(
    fig,
    "Does the task need the J-space? Remove it at every position, layers " + "/".join(map(str, layers)),
    f"{MODEL_NAME}, one item per family. 'Last step alone': a lookup in the prompt for '… → prompt', a recalled fact "
    "for '… → memory'. Controls: the same removal turned by a random rotation, and a fit over a random dictionary.",
    [Patch(color=col, label=cond) for cond, col in CONDS],
)

z_abl = {(r["family"].removeprefix("country-"), r["id"], r["kind"], r["condition"]): r for r in zach["ablate"]}
z_cond = {"nothing removed": "none", "J-space removed": "jspace", "same-size rotated removal": "rotated",
          "random-dictionary removal": "random"}  # fmt: skip
z_kind = {"whole question": "clean", "last step alone": "hop"}
body = []
for (f, kind, cond), (ok, lp) in knock.items():
    z = z_abl.get((f, cases[f]["id"], z_kind[kind], z_cond[cond]))
    body.append([LABEL[f], kind, cond, "✓" if ok else "✗", f"{lp:+.2f}", ("✓" if z["correct"] else "✗") if z else "-",
                 f"{z['logp']:+.2f}" if z else "-"])  # fmt: skip
table(["family", "prompt", "condition", "top token right", "log p", "Zach: right", "Zach: log p"], body)

## 8. This scene next to Zach's family-level numbers

Zach's numbers are means over n=60–150 items (UNVERIFIED); "here" is one item. One item differing from a family mean is expected; what matters is that the same items give the same numbers (the agreement tables in each section).

In [ ]:
rw_summary = summary("read_write", R["read_write"]).get("families", {})
abl_summary = summary("ablate_pilot", R["ablate"]).get("families", {})
body = []
for f in FAMILIES:
    zr, za = rw_summary.get(f, {}), abl_summary.get(f"country-{f}/clean" if f in COUNTRIES else f"{f}/clean", {})
    zc = z_parts_summary.get(f, {}).get("frac", {}).get("jspace", {}).get("concentration")
    lens_flips = max(flips_at(single, f, part=f"lens_x{sc:g}") for sc in SCALES)
    z_flips = max(
        (zr.get("write", {}).get(f"single/lens_x{sc:g}", {}).get("flips", float("nan")) for sc in SCALES),
        default=float("nan"),
    )
    p_none, p_j = (math.exp(knock[f, "whole question", cond][1]) for cond in ("nothing removed", "J-space removed"))
    body.append([
        LABEL[f],
        f"{min(reads[f, la][0] or 10**9 for la in R['window'])} / {zr.get('read_top10', float('nan')):.0%}",
        f"{effect_per_size(f, 'jspace')} / " + (f"{zc[0]:.2f}" if zc else "-"),
        f"{lens_flips:.0%} / {z_flips:.0%}",
        f"{p_none:.0%} → {p_j:.0%} / " + (f"{za['accuracy']['none']:.0%} → {za['accuracy']['jspace']:.0%}" if za else "-"),
    ])  # fmt: skip
table(
    ["family", "read: best bridge rank here / Zach's share in top 10", "J-space effect ÷ size: here / Zach",
     "lens write, best flip rate over strengths: here / Zach", "knock-out: p(answer) here / Zach's accuracy"],
    body,
)  # fmt: skip

**How to read it.** The picture C1/C2 rest on (Qwen3.6-27B, the animal suite): in **memory → prompt** the lens **reads** the bridge (§3), but a push along its direction does **not write** it in the carry window (§6), the patch's J-space part does less than its size predicts (§5), and removing the J-space does not break the task (§7). In **memory → memory** the bridge is read, written, and the task breaks without the J-space. The countries are the positive control: read, written at 1×, effect ÷ size well above 1.

**What one item cannot tell.** Whether this is the rule rather than this item: only the n and CIs in `results/` say that. Next checks (Dan: depth): the statement's positions vs the question's (§4 patches only question tokens), attention knock-out from the question to the statement, and whether the early writing in memory → prompt (§6, early layers) is the same as editing the input word.

Change `SEED` in the first code cell for another scene; `BJ_TARGET=qwen3.5-9b` for the 9B (fewer comparisons: Zach has no knock-out or country runs there).